In [6]:
import cv2
import torch
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
import torch.nn as nn

# Define a simple CNN model
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)

        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):

        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

model = SimpleCNN()

model.load_state_dict(
    torch.load("my_model.pth")
)

# model.eval()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

img = cv2.imread("./scene2.png")

for y in range(0, img.shape[0] - 64, 64):
    for x in range(0, img.shape[1] - 64, 64):

        crop = img[y:y+64, x:x+64]

        crop_rgb = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )

        crop_pil = Image.fromarray(crop_rgb)

        tensor = transform(crop_pil)
        tensor = tensor.unsqueeze(0)

        with torch.no_grad():
            output = model(tensor)
            # print(f"Output for crop at ({x}, {y}): {output}")
            probs = F.softmax(output, dim=1)

        prediction = output.argmax(1).item()
        confidence = probs[0, prediction].item()

        # if prediction == 0:
        if prediction == 2 and confidence > 0.999:
            cv2.rectangle(
                img,
                (x, y),
                (x+64, y+64),
                (0, 255, 0),
                2
            )

        # print(f"Prediction for crop at ({x}, {y}): {prediction}")
        # print(f"Confidence for crop at ({x}, {y}): {confidence}")

cv2.imshow("Detected Label 0", img)
cv2.waitKey(0)
cv2.destroyAllWindows()